In [27]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_core.prompts import PromptTemplate

import uuid

import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# CONFIG 
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data')
PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "banking_docs"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

In [24]:
loader = DirectoryLoader(
    DATA_PATH, 
    glob="*.pdf", 
    loader_cls=PyPDFLoader,
    show_progress=True
)
docs = await loader.aload()
print(docs[0].page_content[:100])
print(docs[0].metadata)

100%|██████████| 6/6 [00:17<00:00,  2.98s/it]

FAQs on “All you wanted to know about NBFCs” 
 
(Updated as on April 29, 2026) 
A. Definitions 
 
1.
{'producer': 'Adobe PDF Library 26.1.183', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '2026-04-29T17:34:59+05:30', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-04-29T17:35:18+05:30', 'sourcemodified': 'D:20260429120322', 'subject': '', 'title': '', 'source': 'e:\\bankChatBot\\data\\ALLNBFC23042025.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}


## Text Splitting

In [25]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""]
)

all_chunks = []
for doc in docs:
    # Enrich metadata for filtering later
    doc.metadata.update({
        "source": doc.metadata.get("source", "unknown"),
        "doc_type": "pdf",
        "total_pages": len(docs)  # rough estimate
    })
    chunks = splitter.split_documents([doc])
    all_chunks.extend(chunks)

print(f" Total chunks: {len(all_chunks)}")

Total chunks: 614
First chunk metadata: {'producer': 'Adobe PDF Library 26.1.183', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '2026-04-29T17:34:59+05:30', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-04-29T17:35:18+05:30', 'sourcemodified': 'D:20260429120322', 'subject': '', 'title': '', 'source': 'e:\\bankChatBot\\data\\ALLNBFC23042025.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}


## Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# Check if DB already has data (prevent duplicates)
if os.path.exists(PERSIST_DIR) and os.listdir(PERSIST_DIR):
    vectorestore = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embedding,
        collection_name=COLLECTION_NAME
    )
    print(" Loaded existing vectorstore")
else:
    ids = [str(uuid.uuid4()) for _ in all_chunks]
    
    vectorestore = Chroma.from_documents(
        documents=all_chunks,
        embedding=embedding,
        persist_directory=PERSIST_DIR,
        collection_name=COLLECTION_NAME,
        ids=ids
    )
    vectorestore.persist()  # Force save
    print(" Created & persisted new vectorstore")

Created new vectorstore


## Similarity search

In [36]:
vectorestore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}
    )

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001AC6FD1CAD0>, search_kwargs={'k': 5})